<a href="https://colab.research.google.com/github/lidwaa/kafka_exemple/blob/main/test_app_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === flagship_app (1).py ===
import streamlit as st
from db import init_db, q1
from pages import (
    render_login_page,
    render_list_page,
    render_flagship_detail_page,
    render_flagship_form_page,
    render_usecase_detail_page,
    render_usecase_form_page,
    render_country_form_page,
    render_users_admin_page,
    render_audit_history_page
)

st.set_page_config(page_title="Flagship Registry", layout="centered")

# Initialize database
init_db()

# Initialize session state variables
if "logged_in" not in st.session_state:
    st.session_state.logged_in = False
if "filter_countries" not in st.session_state:
    st.session_state.filter_countries = []
if "filter_divisions" not in st.session_state:
    st.session_state.filter_divisions = []
if "filter_statuses" not in st.session_state:
    st.session_state.filter_statuses = []

# Login validation
if not st.session_state.logged_in:
    render_login_page()
    st.stop()

# ── Sidebar nav ───────────────────────────────────────────────────────────────
with st.sidebar:
    st.write(f"👤 **{st.session_state.username}**")
    if st.button("🏠 Accueil"):
        st.session_state.page = "list"
        st.rerun()

    if st.session_state.username == "admin":
        if st.button("⚙️ Paramètres Habilitations"):
            st.session_state.page = "users_admin"
            st.rerun()
        if st.button("📜 Historique des Actions"):
            st.session_state.page = "audit_history"
            st.rerun()

    if st.session_state.get("page") in ("flagship_detail", "flagship_form", "usecase_detail", "usecase_form", "country_form"):
        fid = st.session_state.get("current_flagship_id")
        f = q1("SELECT name FROM flagships WHERE id=?", fid) if fid else None
        if f and st.button(f"↩ {f['name']}"):
            st.session_state.page = "flagship_detail"
            st.rerun()
    if st.session_state.get("page") in ("usecase_detail", "usecase_form", "country_form"):
        ucid = st.session_state.get("current_usecase_id")
        uc = q1("SELECT name FROM use_cases WHERE id=?", ucid) if ucid else None
        if uc and st.button(f"↩ {uc['name']}"):
            st.session_state.page = "usecase_detail"
            st.rerun()
    st.divider()
    if st.button("🚪 Déconnexion"):
        st.session_state.clear()
        st.rerun()

# ── Routing ───────────────────────────────────────────────────────────────────
page = st.session_state.get("page", "list")

if page == "list":
    render_list_page()
elif page == "flagship_detail":
    render_flagship_detail_page()
elif page == "flagship_form":
    render_flagship_form_page()
elif page == "usecase_detail":
    render_usecase_detail_page()
elif page == "usecase_form":
    render_usecase_form_page()
elif page == "country_form":
    render_country_form_page()
elif page == "users_admin":
    render_users_admin_page()
elif page == "audit_history":
    render_audit_history_page()


In [ ]:
# === db.py ===
import sqlite3
from pathlib import Path

DB_PATH = Path(__file__).parent / "flagship.db"

def get_db():
    conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    from auth import hash_pw  # lazy import to avoid circular dependency
    conn = get_db()
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL
    );
    CREATE TABLE IF NOT EXISTS flagships (
        id TEXT PRIMARY KEY, rank INTEGER, name TEXT NOT NULL,
        description TEXT, sponsor TEXT, owner TEXT,
        division TEXT, subdivision TEXT, poc TEXT, analyst TEXT
    );
    CREATE TABLE IF NOT EXISTS use_cases (
        id TEXT PRIMARY KEY, flagship_id TEXT NOT NULL,
        name TEXT NOT NULL, description TEXT,
        pole TEXT, entity TEXT, ai_act TEXT, technology TEXT,
        platform TEXT, brick TEXT, domain TEXT, category TEXT,
        sourcing TEXT, ia_value REAL DEFAULT 0,
        contributors TEXT, pilot TEXT, itsvc TEXT,
        date_itsvc TEXT, gate TEXT, date_gate TEXT
    );
    CREATE TABLE IF NOT EXISTS countries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        use_case_id TEXT NOT NULL, country TEXT NOT NULL,
        year INTEGER, status TEXT, prod_date TEXT,
        val_capped REAL DEFAULT 0, val_nbi REAL DEFAULT 0,
        val_gcs REAL DEFAULT 0, val_ca REAL DEFAULT 0,
        val_cor REAL DEFAULT 0, val_rwa REAL DEFAULT 0,
        measure TEXT, rollout TEXT, mrr_cat TEXT, mrr TEXT, owner TEXT,
        UNIQUE(use_case_id, country)
    );
    CREATE TABLE IF NOT EXISTS user_countries (
        username TEXT NOT NULL,
        country_id INTEGER NOT NULL,
        PRIMARY KEY(username, country_id),
        FOREIGN KEY(country_id) REFERENCES countries(id) ON DELETE CASCADE
    );
    CREATE TABLE IF NOT EXISTS audit_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        username TEXT NOT NULL,
        action TEXT NOT NULL,
        table_name TEXT NOT NULL,
        record_id TEXT NOT NULL,
        details TEXT NOT NULL
    );
    """)
    conn.commit()

    if conn.execute("SELECT COUNT(*) FROM users").fetchone()[0] == 0:
        conn.execute("INSERT INTO users (username, password_hash) VALUES (?,?)",
                     ("admin", hash_pw("adminflagship2026")))
        conn.commit()

    if conn.execute("SELECT COUNT(*) FROM flagships").fetchone()[0] == 0:
        seed(conn)

    conn.close()

def seed(conn):
    flagships = [
        ("F01",1,"KYC Acceleration","Automated KYC document review and risk scoring.","Group Compliance","M. Lefevre","GBIS","Financial Crime","A. Moreau","C. Dubois"),
        ("F02",2,"Credit Decision Copilot","Generative AI copilot for credit analysts.","GLBA","J. Bernard","Corporate & Investment Banking","Credit Risk","P. Lambert","L. Garnier"),
        ("F03",3,"Markets AI Trading Assistant","Real-time AI assistant for FX and rates traders.","Global Markets","S. Nakamura","Global Markets","FX & Rates","R. Chen","A. Petit"),
        ("F04",4,"Retail Customer 360","Unified customer view and next-best-action engine.","Retail Banking","C. Martin","Retail Banking","Customer Experience","V. Roux","M. Faure"),
        ("F05",5,"AML Transaction Monitoring","Next-gen anti-money laundering alerting.","Group Compliance","A. Kovac","GBIS","AML","N. Olsen","T. Weiss"),
        ("F06",6,"IT Ops AI","AIOps platform for incident prediction and routing.","Group IT","D. Klein","Group IT","Infrastructure","E. Tran","O. Mendes"),
        ("F07",7,"ESG Intelligence","AI-driven ESG scoring and controversy detection.","Sustainability","I. Aliyev","Group Strategy","Sustainability","B. Costa","H. Park"),
        ("F08",8,"Document Intelligence Hub","Centralized document understanding service.","Operations","F. Boucher","GBIS","Operations","K. Iyer","D. Souza"),
        ("F09",9,"Fraud Shield","Real-time fraud detection across cards and transfers.","Risk","R. Hoffmann","Retail Banking","Fraud","L. Yamada","P. Silva"),
        ("F10",10,"HR Talent AI","AI-assisted CV screening and internal mobility.","Group HR","V. Almeida","Group HR","Talent","S. Kone","E. Ozturk"),
        ("F11",11,"Treasury & ALM Optimizer","AI-driven liquidity forecasting.","Treasury","M. Eriksson","Finance","ALM","T. Andersson","J. Lindqvist"),
        ("F12",12,"Wealth Advisor Copilot","Generative AI assistant for private bankers.","Private Banking","L. Fontaine","Wealth Management","Private Banking","M. Yilmaz","N. Rahman"),
        ("F13",13,"Insurance Claims AI","Automated claims triage and fraud detection.","Insurance","P. Novak","Insurance","Claims","A. Vargas","C. Romero"),
        ("F14",14,"Regulatory Reporting AI","Automated extraction of regulatory reports.","Finance","B. Vargas","Finance","Regulatory Reporting","M. Diallo","V. Ahmed"),
        ("F15",15,"Cybersecurity AI","AI-powered SOC: threat hunting, phishing detection.","Group Security","T. Berg","Group IT","Cybersecurity","H. Mensah","O. Karimov"),
        ("F16",16,"Climate Stress Tests AI","AI models for climate scenario analysis.","Risk","C. Iversen","Risk","Climate Risk","A. Petrov","L. Singh"),
        ("F17",17,"Customer Service Voice AI","Voice AI for inbound retail call centers.","Retail Banking","I. Renaud","Retail Banking","Customer Service","D. Singh","R. Costa"),
        ("F18",18,"Data Governance AI","AI for metadata enrichment and data quality.","CDO Office","S. Andrade","Group Data","Data Governance","F. Costa","M. Khan"),
    ]
    conn.executemany("INSERT OR IGNORE INTO flagships (id, rank, name, description, sponsor, owner, division, subdivision, poc, analyst) VALUES (?,?,?,?,?,?,?,?,?,?)", flagships)

    use_cases = [
        ("U01F01","F01","Document Extraction Pipeline","Extract KYC fields from documents.","Operations","SG Group","High Risk","LLM + OCR","Azure ML","Document AI","NLP","Information Extraction","Hybrid",4200,"AI4Compliance","France","ITSVC-1042","2024-03-15","Gate 4","2024-09-20"),
        ("U02F01","F01","Risk Scoring Engine","PEP/sanction screening and risk scoring.","Compliance","SG Group","High Risk","Gradient Boosting","Databricks","Predictive AI","Tabular ML","Classification","In-house",3100,"Risk Analytics","France","ITSVC-1078","2024-06-10","Gate 3","2025-01-12"),
        ("U01F02","F02","Credit Memo Summarization","Summarize credit memos into key risk factors.","Front Office","GLBA","Limited Risk","LLM (GPT-4)","Internal LLM Gateway","Generative AI","NLP","Summarization","Vendor",5400,"GenAI Lab","France","ITSVC-1130","2024-09-01","Gate 4","2025-02-28"),
        ("U01F03","F03","News Triage Engine","Cluster and score market-moving news.","Trading Floor","GLBA Markets","Limited Risk","Transformers","AWS SageMaker","NLP Pipeline","NLP","Information Retrieval","In-house",6800,"Markets AI Squad","France","ITSVC-1201","2024-11-12","Gate 4","2025-04-08"),
        ("U01F04","F04","Next-Best-Action Recommender","Personalized product recommendations.","Retail","SG Retail FR","Limited Risk","Two-Tower DL","GCP Vertex AI","Recommendation","Recsys","Personalization","In-house",4900,"Personalization Lab","France","ITSVC-1156","2024-08-20","Gate 4","2025-01-30"),
        ("U01F05","F05","Graph-Based Alert Triage","GNN-based suspicious activity scoring.","Compliance","SG Group","High Risk","Graph Neural Networks","Databricks","Predictive AI","Graph ML","Anomaly Detection","In-house",5200,"Graph AI Team","France","ITSVC-1210","2024-12-01","Gate 3","2025-03-22"),
        ("U01F06","F06","Incident Auto-Routing","Classify and route IT tickets.","IT Ops","SG Group","Minimal Risk","BERT Classifier","Azure ML","NLP Classifier","NLP","Classification","In-house",1800,"AIOps Squad","France","ITSVC-1067","2024-05-10","Gate 4","2024-10-15"),
        ("U01F07","F07","ESG Controversy Detection","Detect ESG controversies from news.","Sustainability","SG Group","Limited Risk","LLM + Web Scraping","AWS","NLP Pipeline","NLP","Classification","Hybrid",2200,"ESG AI Team","France","ITSVC-1188","2024-10-05","Gate 3","2025-02-18"),
        ("U01F08","F08","Contract Clause Extraction","Extract key clauses from contracts.","Legal Ops","SG Group","Limited Risk","LLM Fine-tuned","Internal LLM Gateway","Document AI","NLP","Information Extraction","In-house",3700,"DocAI Team","France","ITSVC-1145","2024-07-22","Gate 4","2024-12-15"),
        ("U01F09","F09","Card Fraud Real-Time Scoring","Sub-100ms fraud scoring on card authorizations.","Fraud","SG Retail FR","High Risk","Gradient Boosting","On-prem GPU","Predictive AI","Tabular ML","Anomaly Detection","In-house",6100,"Fraud AI Team","France","ITSVC-1098","2024-06-30","Gate 4","2024-11-20"),
        ("U01F10","F10","Internal Mobility Matcher","Recommend internal job opportunities.","HR","SG Group","Limited Risk","Sentence Transformers","Azure ML","Embedding Search","NLP","Matching","In-house",1500,"HR Analytics","France","ITSVC-1175","2024-09-18","Gate 3","2025-02-05"),
        ("U01F11","F11","Liquidity Forecasting","30-day rolling liquidity forecast.","Treasury","SG Group","Minimal Risk","Time Series DL","Databricks","Forecasting","Time Series","Forecasting","In-house",2800,"Treasury AI","France","ITSVC-1163","2024-08-10","Gate 3","2025-01-15"),
        ("U01F12","F12","Portfolio Commentary Generator","Auto-generate monthly client portfolio reviews.","Private Bank","SG PB","Limited Risk","LLM (GPT-4)","Internal LLM Gateway","Generative AI","NLP","Generation","Vendor",3300,"GenAI Lab","Luxembourg","ITSVC-1192","2024-10-20","Gate 3","2025-03-10"),
        ("U01F13","F13","Claims Triage","Auto-classify insurance claims at intake.","Claims","SG Insurance","High Risk","Multimodal LLM","Azure ML","Document AI","Multimodal","Classification","Hybrid",2600,"Insurance AI","France","ITSVC-1207","2024-11-25","Gate 3","2025-04-02"),
        ("U01F14","F14","COREP Auto-Validation","Validate COREP submissions with anomaly detection.","Finance","SG Group","High Risk","Anomaly Detection ML","On-prem","Predictive AI","Tabular ML","Anomaly Detection","In-house",2900,"RegReporting AI","France","ITSVC-1199","2024-11-08","Gate 3","2025-03-30"),
        ("U01F15","F15","Phishing Email Detection","Detect phishing in inbound corporate email.","CyberSec","SG Group","Limited Risk","Multimodal Classifier","Azure ML","Multimodal AI","Multimodal","Classification","In-house",2400,"CyberAI Squad","France","ITSVC-1182","2024-09-30","Gate 4","2025-02-22"),
        ("U01F16","F16","Physical Risk Scoring","Score real-estate portfolio climate risk.","Climate Risk","SG Group","Limited Risk","Geo-spatial ML","GCP Vertex AI","Predictive AI","Geospatial","Scoring","In-house",1900,"Climate AI","France","ITSVC-1218","2024-12-15","Gate 2","2025-04-25"),
        ("U01F17","F17","Post-Call Summarization","Auto-summarize agent-customer calls.","Customer Service","SG Retail FR","Limited Risk","Speech-to-Text + LLM","Azure","Speech AI","Speech","Transcription","Vendor",2100,"Voice AI Team","France","ITSVC-1213","2024-12-05","Gate 3","2025-04-12"),
        ("U01F18","F18","Auto Data Cataloging","Auto-classify and tag datasets.","Data Governance","SG Group","Minimal Risk","LLM + Schema Inference","Internal LLM Gateway","Generative AI","NLP","Classification","In-house",1700,"CDO Data Team","France","ITSVC-1224","2025-01-15","Gate 2","2025-04-30"),
    ]
    conn.executemany("INSERT OR IGNORE INTO use_cases (id, flagship_id, name, description, pole, entity, ai_act, technology, platform, brick, domain, category, sourcing, ia_value, contributors, pilot, itsvc, date_itsvc, gate, date_gate) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", use_cases)

    countries_data = [
        ("U01F01","France",2025,"Industrialized","2024-10-01",2800,0,2200,600,0,0,"Measured","Local","Operational Efficiency","MRR-FR-2024-018","M. Lefevre"),
        ("U01F01","Italy",2025,"Pilot","2025-03-15",900,0,700,200,0,0,"Estimated","Local","Operational Efficiency","MRR-IT-2025-003","G. Rossi"),
        ("U02F01","France",2025,"Pilot","2025-02-01",1500,0,0,0,1500,0,"Estimated","Local","Risk Management","MRR-FR-2025-006","M. Lefevre"),
        ("U01F02","France",2025,"Industrialized","2025-03-01",3200,1800,1400,0,0,0,"Measured","Global","Productivity","MRR-FR-2025-011","J. Bernard"),
        ("U01F02","Luxembourg",2025,"Pilot","2025-05-15",600,400,200,0,0,0,"Estimated","Local","Productivity","MRR-LU-2025-002","F. Schmit"),
        ("U01F03","France",2025,"Industrialized","2025-04-10",4500,4500,0,0,0,0,"Measured","Global","Revenue","MRR-FR-2025-019","S. Nakamura"),
        ("U01F04","France",2025,"Industrialized","2025-02-15",3400,3400,0,0,0,0,"Measured","Local","Revenue","MRR-FR-2025-009","C. Martin"),
        ("U01F05","France",2025,"Pilot","2025-05-01",2100,0,0,0,2100,0,"Estimated","Local","Risk Management","MRR-FR-2025-014","A. Kovac"),
        ("U01F06","France",2025,"Industrialized","2024-11-01",1400,0,1400,0,0,0,"Measured","Global","Operational Efficiency","MRR-FR-2024-029","D. Klein"),
    ]
    conn.executemany("INSERT OR IGNORE INTO countries (use_case_id,country,year,status,prod_date,val_capped,val_nbi,val_gcs,val_ca,val_cor,val_rwa,measure,rollout,mrr_cat,mrr,owner) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", countries_data)
    conn.commit()

def q(sql, *args):
    conn = get_db()
    rows = conn.execute(sql, args).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def q1(sql, *args):
    conn = get_db()
    r = conn.execute(sql, args).fetchone()
    conn.close()
    return dict(r) if r else None

def run(sql, *args):
    conn = get_db()
    conn.execute(sql, args)
    conn.commit()
    conn.close()

def next_f_id():
    r = q1("SELECT MAX(CAST(SUBSTR(id,2) AS INT)) n FROM flagships WHERE id LIKE 'F%'")
    return f"F{(r['n'] or 0)+1:02d}"

def next_uc_id(fid):
    n = q1("SELECT COUNT(*) n FROM use_cases WHERE flagship_id=?", fid)["n"]
    return f"U{n+1:02d}{fid}"

def get_table_columns(table_name):
    conn = get_db()
    cursor = conn.execute(f"PRAGMA table_info({table_name})")
    cols = [r["name"] for r in cursor.fetchall()]
    conn.close()
    return cols

def sanitize_col_name(name):
    import re
    s = name.strip().lower()
    s = re.sub(r'\s+', '_', s)
    s = re.sub(r'[^a-z0-9_]', '', s)
    return s


In [ ]:
# === auth.py ===
import hashlib
from db import q1, run

def hash_pw(pw):
    return hashlib.sha256(pw.encode()).hexdigest()

def check_login(u, p):
    return bool(q1("SELECT 1 FROM users WHERE username=? AND password_hash=?", u, hash_pw(p)))

def create_user(u, p):
    try:
        run("INSERT INTO users (username, password_hash) VALUES (?,?)", u, hash_pw(p))
        return True
    except:
        return False

def is_flagship_authorized(username, flagship_id):
    if username == "admin":
        return True
    return bool(q1("""
        SELECT 1 FROM user_countries uc_auth
        JOIN countries c ON uc_auth.country_id = c.id
        JOIN use_cases uc ON c.use_case_id = uc.id
        WHERE uc_auth.username = ? AND uc.flagship_id = ?
    """, username, flagship_id))

def is_usecase_authorized(username, usecase_id):
    if username == "admin":
        return True
    return bool(q1("""
        SELECT 1 FROM user_countries uc_auth
        JOIN countries c ON uc_auth.country_id = c.id
        WHERE uc_auth.username = ? AND c.use_case_id = ?
    """, username, usecase_id))

def is_country_authorized(username, country_id):
    if username == "admin":
        return True
    return bool(q1("SELECT 1 FROM user_countries WHERE username=? AND country_id=?", username, country_id))

def log_change(username, action, table_name, record_id, details):
    run("""
        INSERT INTO audit_logs (username, action, table_name, record_id, details)
        VALUES (?,?,?,?,?)
    """, username, action, table_name, str(record_id), details)


In [ ]:
# === ui_components.py ===
import streamlit as st

COUNTRIES = ['France','Italy','Belgium','Luxembourg','Poland','Czech Republic','Romania',
             'Morocco','Algeria','Tunisia','Germany','Spain','UK','Switzerland','Netherlands','USA','Singapore','Japan']
AI_ACT    = ["Minimal Risk","Limited Risk","High Risk","Unacceptable Risk"]
STATUS    = ["Draft","Pilot","Industrialized"]
MEASURE   = ["Estimated","Measured"]
ROLLOUT   = ["Local","Regional","Global"]
GATES     = ["Gate 1","Gate 2","Gate 3","Gate 4","Gate 5"]
SOURCING  = ["In-house","Vendor","Hybrid"]
PLATFORMS = ["Azure ML","GCP Vertex AI","AWS SageMaker","Databricks","Internal LLM Gateway","On-prem","Azure","AWS","Other"]
MRR_CATS  = ["","Operational Efficiency","Risk Management","Revenue","Productivity","Cost Reduction"]

PROTECTED_COLS = {
    "flagships": ["id", "name", "rank"],
    "use_cases": ["id", "flagship_id", "name"],
    "countries": ["id", "use_case_id", "country"]
}

SELECT_BOXES = {
    "ai_act": AI_ACT,
    "platform": PLATFORMS,
    "sourcing": SOURCING,
    "gate": GATES,
    "status": STATUS,
    "measure": MEASURE,
    "rollout": ROLLOUT,
    "mrr_cat": MRR_CATS
}

def render_field_input(table_name, col_name, value, key_prefix):
    if col_name in SELECT_BOXES:
        options = SELECT_BOXES[col_name]
        try:
            idx = options.index(value)
        except ValueError:
            idx = 0
        return st.selectbox(col_name.replace("_", " ").title(), options, index=idx, key=f"{key_prefix}_{col_name}")
    elif col_name == "description":
        return st.text_area("Description", value=value or "", key=f"{key_prefix}_{col_name}")
    elif col_name in ("ia_value", "val_capped", "val_nbi", "val_gcs", "val_ca", "val_cor", "val_rwa"):
        return st.number_input(col_name.replace("_", " ").title(), min_value=0, value=int(value or 0), key=f"{key_prefix}_{col_name}")
    elif col_name == "year":
        return st.number_input("Année", min_value=2020, max_value=2030, value=int(value or 2025), key=f"{key_prefix}_{col_name}")
    elif col_name == "rank":
        return st.number_input("Rang", min_value=1, max_value=99, value=int(value or 1), key=f"{key_prefix}_{col_name}")
    else:
        return st.text_input(col_name.replace("_", " ").title(), value=str(value) if value is not None else "", key=f"{key_prefix}_{col_name}")

def render_details_dynamic(row, table_name, exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = []
    cols = [k for k in row.keys() if k not in exclude_cols]
    if not cols:
        return
    col1, col2 = st.columns(2)
    for i, col_name in enumerate(cols):
        val = row[col_name]
        if val is None:
            display_val = "—"
        elif isinstance(val, (int, float)) and col_name in ("ia_value", "val_capped", "val_nbi", "val_gcs", "val_ca", "val_cor", "val_rwa"):
            display_val = f"{int(val):,} k€"
        else:
            display_val = str(val)
        label = col_name.replace("_", " ").title()
        with col1 if i % 2 == 0 else col2:
            st.write(f"**{label}** : {display_val}")


In [ ]:
# === pages.py ===
import streamlit as st
from db import q, q1, run, next_f_id, next_uc_id, get_table_columns, sanitize_col_name
from auth import is_flagship_authorized, is_usecase_authorized, is_country_authorized, log_change, check_login, create_user
from ui_components import PROTECTED_COLS, COUNTRIES, render_field_input, render_details_dynamic

def render_login_page():
    st.title("Flagship Registry")
    st.subheader("Connexion")
    u = st.text_input("Utilisateur", key="li_u")
    p = st.text_input("Mot de passe", type="password", key="li_p")
    if st.button("Se connecter", type="primary"):
        if check_login(u, p):
            st.session_state.logged_in = True
            st.session_state.username = u
            st.session_state.page = "list"
            st.rerun()
        else:
            st.error("Identifiants incorrects.")
    st.caption("Compte par défaut : admin / admin123")

def render_list_page():
    st.title("Flagship Registry")

    # Only visible to admin
    if st.session_state.username == "admin":
        f_count = q1("SELECT COUNT(*) n FROM flagships")["n"]
        uc_count = q1("SELECT COUNT(*) n FROM use_cases")["n"]
        val = q1("SELECT COALESCE(SUM(ia_value),0) n FROM use_cases")["n"]
        c_count = q1("SELECT COUNT(DISTINCT country) n FROM countries")["n"]

        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Flagships", f_count)
        col2.metric("Use Cases", uc_count)
        col3.metric("IA Value (k€)", f"{int(val):,}")
        col4.metric("Pays", c_count)
        st.divider()

    search = st.text_input("Rechercher", placeholder="Rechercher...")

    # Admin Advanced Filters
    if st.session_state.username == "admin":
        with st.expander("🔍 Filtres Avancés"):
            c_f1, c_f2, c_f3 = st.columns(3)
            with c_f1:
                all_countries = [r["country"] for r in q("SELECT DISTINCT country FROM countries ORDER BY country")]
                st.session_state.filter_countries = st.multiselect(
                    "Filtrer par Pays",
                    all_countries,
                    default=st.session_state.filter_countries
                )
            with c_f2:
                all_divisions = [r["division"] for r in q("SELECT DISTINCT division FROM flagships WHERE division IS NOT NULL AND division != '' ORDER BY division")]
                st.session_state.filter_divisions = st.multiselect(
                    "Filtrer par Division",
                    all_divisions,
                    default=st.session_state.filter_divisions
                )
            with c_f3:
                all_statuses = [r["status"] for r in q("SELECT DISTINCT status FROM countries WHERE status IS NOT NULL AND status != '' ORDER BY status")]
                st.session_state.filter_statuses = st.multiselect(
                    "Filtrer par Statut",
                    all_statuses,
                    default=st.session_state.filter_statuses
                )

    # Load authorized flagships
    if st.session_state.username == "admin":
        flagships = q("SELECT * FROM flagships ORDER BY rank")
        # Apply filters in Python for admin
        if st.session_state.filter_countries:
            flagships = [
                f for f in flagships
                if q1("""
                    SELECT 1 FROM countries c
                    JOIN use_cases uc ON c.use_case_id = uc.id
                    WHERE uc.flagship_id = ? AND c.country IN ({})
                """.format(", ".join(["?"] * len(st.session_state.filter_countries))), f["id"], *st.session_state.filter_countries)
            ]
        if st.session_state.filter_divisions:
            flagships = [f for f in flagships if f.get("division") in st.session_state.filter_divisions]
        if st.session_state.filter_statuses:
            flagships = [
                f for f in flagships
                if q1("""
                    SELECT 1 FROM countries c
                    JOIN use_cases uc ON c.use_case_id = uc.id
                    WHERE uc.flagship_id = ? AND c.status IN ({})
                """.format(", ".join(["?"] * len(st.session_state.filter_statuses))), f["id"], *st.session_state.filter_statuses)
            ]
    else:
        flagships = q("""
            SELECT DISTINCT f.* FROM flagships f
            JOIN use_cases uc ON f.id = uc.flagship_id
            JOIN countries c ON uc.id = c.use_case_id
            JOIN user_countries uc_auth ON c.id = uc_auth.country_id
            WHERE uc_auth.username = ?
            ORDER BY f.rank
        """, st.session_state.username)

    if search:
        s = search.lower()
        flagships = [
            f for f in flagships
            if any(s in str(v).lower() for v in f.values() if v is not None)
        ]

    # Only admin can create flagships
    if st.session_state.username == "admin":
        if st.button("＋ Nouveau flagship", type="primary"):
            st.session_state.page = "flagship_form"
            st.session_state.edit_flagship_id = None
            st.rerun()
        st.divider()

    for f in flagships:
        if st.session_state.username == "admin":
            if st.session_state.filter_countries or st.session_state.filter_statuses:
                sql = """
                    SELECT COUNT(DISTINCT uc.id) n FROM use_cases uc
                    JOIN countries c ON uc.id = c.use_case_id
                    WHERE uc.flagship_id = ?
                """
                params = [f["id"]]
                if st.session_state.filter_countries:
                    sql += " AND c.country IN ({})".format(", ".join(["?"] * len(st.session_state.filter_countries)))
                    params.extend(st.session_state.filter_countries)
                if st.session_state.filter_statuses:
                    sql += " AND c.status IN ({})".format(", ".join(["?"] * len(st.session_state.filter_statuses)))
                    params.extend(st.session_state.filter_statuses)
                uc_n = q1(sql, *params)["n"]
            else:
                uc_n = q1("SELECT COUNT(*) n FROM use_cases WHERE flagship_id=?", f["id"])["n"]
        else:
            uc_n = q1("""
                SELECT COUNT(DISTINCT uc.id) n FROM use_cases uc
                JOIN countries c ON uc.id = c.use_case_id
                JOIN user_countries uc_auth ON c.id = uc_auth.country_id
                WHERE uc.flagship_id = ? AND uc_auth.username = ?
            """, f["id"], st.session_state.username)["n"]

        division_str = f"  ·  {f['division']}" if "division" in f and f["division"] else ""
        with st.expander(f"#{f['rank']} — {f['name']}  ·  {uc_n} use case(s){division_str}"):
            if "description" in f and f["description"]:
                st.write(f["description"])

            captions = []
            for k in ("sponsor", "owner", "poc"):
                if k in f and f[k]:
                    captions.append(f"{k.capitalize()}: {f[k]}")
            if captions:
                st.caption("  |  ".join(captions))

            if st.button("Ouvrir", key=f"open_{f['id']}"):
                st.session_state.page = "flagship_detail"
                st.session_state.current_flagship_id = f["id"]
                st.rerun()

def render_flagship_detail_page():
    fid = st.session_state.get("current_flagship_id")
    f = q1("SELECT * FROM flagships WHERE id=?", fid)
    if not f:
        st.error("Flagship introuvable.")
        st.stop()

    if not is_flagship_authorized(st.session_state.username, fid):
        st.error("Accès non autorisé.")
        st.stop()

    st.title(f["name"])
    st.caption(f"#{f['rank']}")
    if "description" in f and f["description"]:
        st.write(f["description"])

    render_details_dynamic(f, "flagships", exclude_cols=["id", "name", "rank", "description"])
    st.write("")

    if st.session_state.username == "admin":
        col_edit, col_del = st.columns(2)
        with col_edit:
            if st.button("✎ Modifier le flagship"):
                st.session_state.page = "flagship_form"
                st.session_state.edit_flagship_id = fid
                st.rerun()
        with col_del:
            if st.button("🗑 Supprimer le flagship"):
                st.session_state["confirm_del_f"] = True
            if st.session_state.get("confirm_del_f"):
                st.warning("Confirmer la suppression ?")
                if st.button("Oui, supprimer", type="primary"):
                    log_change(st.session_state.username, "DELETE", "flagships", fid, f"Suppression du flagship: {f['name']}")
                    run("DELETE FROM flagships WHERE id=?", fid)
                    st.session_state.page = "list"
                    st.session_state.pop("confirm_del_f", None)
                    st.rerun()
    else:
        if st.button("✎ Modifier le flagship"):
            st.session_state.page = "flagship_form"
            st.session_state.edit_flagship_id = fid
            st.rerun()

    st.divider()
    st.subheader("Use Cases")

    # Only admin can add use cases
    if st.session_state.username == "admin":
        if st.button("＋ Ajouter un use case", type="primary"):
            st.session_state.page = "usecase_form"
            st.session_state.edit_usecase_id = None
            st.rerun()

    # Load authorized use cases
    if st.session_state.username == "admin":
        if st.session_state.filter_countries or st.session_state.filter_statuses:
            sql = """
                SELECT DISTINCT uc.* FROM use_cases uc
                JOIN countries c ON uc.id = c.use_case_id
                WHERE uc.flagship_id = ?
            """
            params = [fid]
            if st.session_state.filter_countries:
                sql += " AND c.country IN ({})".format(", ".join(["?"] * len(st.session_state.filter_countries)))
                params.extend(st.session_state.filter_countries)
            if st.session_state.filter_statuses:
                sql += " AND c.status IN ({})".format(", ".join(["?"] * len(st.session_state.filter_statuses)))
                params.extend(st.session_state.filter_statuses)
            sql += " ORDER BY uc.id"
            use_cases = q(sql, *params)
        else:
            use_cases = q("SELECT * FROM use_cases WHERE flagship_id=? ORDER BY id", fid)
    else:
        use_cases = q("""
            SELECT DISTINCT uc.* FROM use_cases uc
            JOIN countries c ON uc.id = c.use_case_id
            JOIN user_countries uc_auth ON c.id = uc_auth.country_id
            WHERE uc.flagship_id = ? AND uc_auth.username = ?
            ORDER BY uc.id
        """, fid, st.session_state.username)

    if not use_cases:
        st.info("Aucun use case accessible pour ce flagship.")
    for uc in use_cases:
        if st.session_state.username == "admin":
            if st.session_state.filter_countries or st.session_state.filter_statuses:
                sql = "SELECT * FROM countries WHERE use_case_id = ?"
                params = [uc["id"]]
                if st.session_state.filter_countries:
                    sql += " AND country IN ({})".format(", ".join(["?"] * len(st.session_state.filter_countries)))
                    params.extend(st.session_state.filter_countries)
                if st.session_state.filter_statuses:
                    sql += " AND status IN ({})".format(", ".join(["?"] * len(st.session_state.filter_statuses)))
                    params.extend(st.session_state.filter_statuses)
                sql += " ORDER BY country"
                countries = q(sql, *params)
            else:
                countries = q("SELECT * FROM countries WHERE use_case_id=? ORDER BY country", uc["id"])
        else:
            countries = q("""
                SELECT c.* FROM countries c
                JOIN user_countries uc_auth ON c.id = uc_auth.country_id
                WHERE c.use_case_id = ? AND uc_auth.username = ?
                ORDER BY c.country
            """, uc["id"], st.session_state.username)

        ai_act_val = uc.get("ai_act", "")
        ai_act_str = f"  ·  {ai_act_val}" if ai_act_val else ""
        with st.expander(f"{uc['name']}{ai_act_str}  ·  {len(countries)} pays"):
            if "description" in uc and uc["description"]:
                st.write(uc["description"])

            cap = []
            if "technology" in uc and uc["technology"]:
                cap.append(f"Tech: {uc['technology']}")
            if "platform" in uc and uc["platform"]:
                cap.append(f"Platform: {uc['platform']}")
            if "gate" in uc and uc["gate"]:
                cap.append(f"Gate: {uc['gate']}")
            if "ia_value" in uc and uc["ia_value"] is not None:
                cap.append(f"IA Value: {int(uc['ia_value']):,} k€")
            if cap:
                st.caption("  |  ".join(cap))

            ca, cb = st.columns(2)
            with ca:
                if st.button("Ouvrir", key=f"o_{uc['id']}"):
                    st.session_state.page = "usecase_detail"
                    st.session_state.current_usecase_id = uc["id"]
                    st.rerun()
            with cb:
                if st.button("✎ Modifier", key=f"e_{uc['id']}"):
                    st.session_state.page = "usecase_form"
                    st.session_state.edit_usecase_id = uc["id"]
                    st.rerun()

def render_flagship_form_page():
    fid = st.session_state.get("edit_flagship_id")
    f = q1("SELECT * FROM flagships WHERE id=?", fid) if fid else {}
    is_new = not bool(f)

    if fid and not is_flagship_authorized(st.session_state.username, fid):
        st.error("Accès non autorisé.")
        st.stop()

    st.title("Nouveau flagship" if is_new else "Modifier le flagship")

    cols = get_table_columns("flagships")
    extra_cols = [c for c in cols if c not in ("id", "name", "rank")]

    form_values = {}
    with st.form("f_form"):
        name = st.text_input("Nom *", value=f.get("name",""))
        rank = st.number_input("Rang", min_value=1, max_value=99, value=int(f.get("rank",1)))

        if extra_cols:
            col1, col2 = st.columns(2)
            for i, col_name in enumerate(extra_cols):
                val = f.get(col_name, "")
                with col1 if i % 2 == 0 else col2:
                    form_values[col_name] = render_field_input("flagships", col_name, val, "fs")

        if st.form_submit_button("💾 Sauvegarder", type="primary"):
            if not name:
                st.error("Le nom est obligatoire.")
            else:
                new_id = fid or next_f_id()
                data = {"id": new_id, "name": name, "rank": rank}
                for col_name in extra_cols:
                    data[col_name] = form_values[col_name]

                if is_new:
                    keys = list(data.keys())
                    placeholders = ", ".join(["?"] * len(keys))
                    sql = f"INSERT INTO flagships ({', '.join(keys)}) VALUES ({placeholders})"
                    run(sql, *list(data.values()))
                    log_change(st.session_state.username, "INSERT", "flagships", new_id, f"Création du flagship: {name}")
                else:
                    keys = [k for k in data.keys() if k != "id"]
                    assignments = ", ".join([f"{k}=?" for k in keys])
                    values = [data[k] for k in keys] + [new_id]
                    sql = f"UPDATE flagships SET {assignments} WHERE id=?"
                    run(sql, *values)
                    log_change(st.session_state.username, "UPDATE", "flagships", new_id, f"Modification du flagship: {name} (Rang: {rank})")

                st.session_state.current_flagship_id = new_id
                st.session_state.page = "flagship_detail"
                st.rerun()

    # Column configuration (admin only)
    if st.session_state.username == "admin":
        st.write("---")
        st.subheader("⚙️ Configurer les colonnes de la table Flagships")
        st.warning("⚠️ ATTENTION : Supprimer une colonne supprimera définitivement les données correspondantes pour tous les flagships.")

        col_add1, col_add2 = st.columns([3, 1])
        with col_add1:
            new_field = st.text_input("Nom du nouveau champ", placeholder="ex: priorite_division, budget...", key="new_field_flagships")
        with col_add2:
            st.write(" ")
            st.write(" ")
            if st.button("➕ Ajouter", key="btn_add_flagships"):
                if new_field:
                    sanitized = sanitize_col_name(new_field)
                    existing = get_table_columns("flagships")
                    if not sanitized:
                        st.error("Nom de champ invalide.")
                    elif sanitized in existing:
                        st.error("Ce champ existe déjà.")
                    else:
                        run(f"ALTER TABLE flagships ADD COLUMN {sanitized} TEXT")
                        log_change(st.session_state.username, "SCHEMA_ADD", "flagships", sanitized, f"Ajout de la colonne: {sanitized}")
                        st.success(f"Champ '{sanitized}' ajouté !")
                        st.rerun()
                else:
                    st.error("Veuillez saisir un nom.")

        st.write("**Champs personnalisables existants :**")
        deletable_cols = [c for c in cols if c not in PROTECTED_COLS["flagships"]]

        if deletable_cols:
            for c in deletable_cols:
                col_list1, col_list2 = st.columns([3, 1])
                with col_list1:
                    st.code(c)
                with col_list2:
                    if st.button("🗑️ Supprimer", key=f"del_flagships_{c}"):
                        run(f"ALTER TABLE flagships DROP COLUMN {c}")
                        log_change(st.session_state.username, "SCHEMA_DROP", "flagships", c, f"Suppression de la colonne: {c}")
                        st.success(f"Champ '{c}' supprimé !")
                        st.rerun()
        else:
            st.info("Aucun champ personnalisable.")

def render_usecase_detail_page():
    ucid = st.session_state.get("current_usecase_id")
    uc = q1("SELECT * FROM use_cases WHERE id=?", ucid)
    if not uc:
        st.error("Use case introuvable.")
        st.stop()

    if not is_usecase_authorized(st.session_state.username, ucid):
        st.error("Accès non autorisé.")
        st.stop()

    st.title(uc["name"])
    st.caption(f"{uc['id']}")
    if "description" in uc and uc["description"]:
        st.write(uc["description"])

    render_details_dynamic(uc, "use_cases", exclude_cols=["id", "flagship_id", "name", "description"])
    st.write("")

    if st.session_state.username == "admin":
        ce, cd = st.columns(2)
        with ce:
            if st.button("✎ Modifier le use case"):
                st.session_state.page = "usecase_form"
                st.session_state.edit_usecase_id = ucid
                st.rerun()
        with cd:
            if st.button("🗑 Supprimer le use case"):
                st.session_state["confirm_del_uc"] = True
            if st.session_state.get("confirm_del_uc"):
                st.warning("Confirmer la suppression ?")
                if st.button("Oui, supprimer", type="primary"):
                    log_change(st.session_state.username, "DELETE", "use_cases", ucid, f"Suppression du use case: {uc['name']}")
                    run("DELETE FROM use_cases WHERE id=?", ucid)
                    st.session_state.page = "flagship_detail"
                    st.session_state.pop("confirm_del_uc", None)
                    st.rerun()
    else:
        if st.button("✎ Modifier le use case"):
            st.session_state.page = "usecase_form"
            st.session_state.edit_usecase_id = ucid
            st.rerun()

    st.divider()
    st.subheader("Déploiements pays")

    # Only admin can add countries
    if st.session_state.username == "admin":
        if st.button("＋ Ajouter un pays", type="primary"):
            st.session_state.page = "country_form"
            st.session_state.edit_country = None
            st.rerun()

    # Load authorized countries
    if st.session_state.username == "admin":
        if st.session_state.filter_countries or st.session_state.filter_statuses:
            sql = "SELECT * FROM countries WHERE use_case_id = ?"
            params = [ucid]
            if st.session_state.filter_countries:
                sql += " AND country IN ({})".format(", ".join(["?"] * len(st.session_state.filter_countries)))
                params.extend(st.session_state.filter_countries)
            if st.session_state.filter_statuses:
                sql += " AND status IN ({})".format(", ".join(["?"] * len(st.session_state.filter_statuses)))
                params.extend(st.session_state.filter_statuses)
            sql += " ORDER BY country"
            countries = q(sql, *params)
        else:
            countries = q("SELECT * FROM countries WHERE use_case_id=? ORDER BY country", ucid)
    else:
        countries = q("""
            SELECT c.* FROM countries c
            JOIN user_countries uc_auth ON c.id = uc_auth.country_id
            WHERE c.use_case_id = ? AND uc_auth.username = ?
            ORDER BY c.country
        """, ucid, st.session_state.username)

    if not countries:
        st.info("Aucun pays configuré ou accessible.")
    for c in countries:
        status_val = c.get("status", "")
        status_str = f"  ·  {status_val}" if status_val else ""

        val_capped_val = c.get("val_capped", 0)
        val_capped_str = f"  ·  {int(val_capped_val):,} k€ capped" if val_capped_val else ""

        with st.expander(f"{c['country']}{status_str}{val_capped_str}"):
            render_details_dynamic(c, "countries", exclude_cols=["id", "use_case_id", "country"])

            st.write("")
            if st.session_state.username == "admin":
                cma, cda = st.columns(2)
                with cma:
                    if st.button("✎ Modifier", key=f"mc_{c['id']}"):
                        st.session_state.page = "country_form"
                        st.session_state.edit_country = c["country"]
                        st.rerun()
                with cda:
                    if st.button("🗑 Supprimer", key=f"dc_{c['id']}"):
                        log_change(st.session_state.username, "DELETE", "countries", f"{ucid}/{c['country']}", f"Suppression du déploiement pays: {c['country']}")
                        run("DELETE FROM countries WHERE use_case_id=? AND country=?", ucid, c["country"])
                        st.rerun()
            else:
                if st.button("✎ Modifier", key=f"mc_{c['id']}"):
                    st.session_state.page = "country_form"
                    st.session_state.edit_country = c["country"]
                    st.rerun()

def render_usecase_form_page():
    ucid = st.session_state.get("edit_usecase_id")
    uc = q1("SELECT * FROM use_cases WHERE id=?", ucid) if ucid else {}
    fid = uc.get("flagship_id") if uc else st.session_state.get("current_flagship_id")
    is_new = not bool(uc)

    if ucid and not is_usecase_authorized(st.session_state.username, ucid):
        st.error("Accès non autorisé.")
        st.stop()

    st.title("Nouveau use case" if is_new else "Modifier le use case")

    cols = get_table_columns("use_cases")
    extra_cols = [c for c in cols if c not in ("id", "flagship_id", "name")]

    form_values = {}
    with st.form("uc_form"):
        name = st.text_input("Nom *", value=uc.get("name",""))

        if extra_cols:
            col1, col2 = st.columns(2)
            for i, col_name in enumerate(extra_cols):
                val = uc.get(col_name, "")
                with col1 if i % 2 == 0 else col2:
                    form_values[col_name] = render_field_input("use_cases", col_name, val, "uc")

        if st.form_submit_button("💾 Sauvegarder", type="primary"):
            if not name:
                st.error("Le nom est obligatoire.")
            else:
                new_id = ucid or next_uc_id(fid)
                data = {"id": new_id, "flagship_id": fid, "name": name}
                for col_name in extra_cols:
                    data[col_name] = form_values[col_name]

                if is_new:
                    keys = list(data.keys())
                    placeholders = ", ".join(["?"] * len(keys))
                    sql = f"INSERT INTO use_cases ({', '.join(keys)}) VALUES ({placeholders})"
                    run(sql, *list(data.values()))
                    log_change(st.session_state.username, "INSERT", "use_cases", new_id, f"Création du use case: {name}")
                else:
                    keys = [k for k in data.keys() if k != "id"]
                    assignments = ", ".join([f"{k}=?" for k in keys])
                    values = [data[k] for k in keys] + [new_id]
                    sql = f"UPDATE use_cases SET {assignments} WHERE id=?"
                    run(sql, *values)
                    log_change(st.session_state.username, "UPDATE", "use_cases", new_id, f"Modification du use case: {name}")

                st.session_state.current_usecase_id = new_id
                st.session_state.page = "flagship_detail"
                st.rerun()

    # Column configuration (admin only)
    if st.session_state.username == "admin":
        st.write("---")
        st.subheader("⚙️ Configurer les colonnes de la table Use Cases")
        st.warning("⚠️ ATTENTION : Supprimer une colonne supprimera définitivement les données correspondantes pour tous les use cases.")

        col_add1, col_add2 = st.columns([3, 1])
        with col_add1:
            new_field = st.text_input("Nom du nouveau champ", placeholder="ex: budget, ref_tech...", key="new_field_usecases")
        with col_add2:
            st.write(" ")
            st.write(" ")
            if st.button("➕ Ajouter", key="btn_add_usecases"):
                if new_field:
                    sanitized = sanitize_col_name(new_field)
                    existing = get_table_columns("use_cases")
                    if not sanitized:
                        st.error("Nom de champ invalide.")
                    elif sanitized in existing:
                        st.error("Ce champ existe déjà.")
                    else:
                        run(f"ALTER TABLE use_cases ADD COLUMN {sanitized} TEXT")
                        log_change(st.session_state.username, "SCHEMA_ADD", "use_cases", sanitized, f"Ajout de la colonne: {sanitized}")
                        st.success(f"Champ '{sanitized}' ajouté !")
                        st.rerun()
                else:
                    st.error("Veuillez saisir un nom.")

        st.write("**Champs personnalisables existants :**")
        deletable_cols = [c for c in cols if c not in PROTECTED_COLS["use_cases"]]

        if deletable_cols:
            for c in deletable_cols:
                col_list1, col_list2 = st.columns([3, 1])
                with col_list1:
                    st.code(c)
                with col_list2:
                    if st.button("🗑️ Supprimer", key=f"del_usecases_{c}"):
                        run(f"ALTER TABLE use_cases DROP COLUMN {c}")
                        log_change(st.session_state.username, "SCHEMA_DROP", "use_cases", c, f"Suppression de la colonne: {c}")
                        st.success(f"Champ '{c}' supprimé !")
                        st.rerun()
        else:
            st.info("Aucun champ personnalisable.")

def render_country_form_page():
    ucid        = st.session_state.get("current_usecase_id")
    edit_c      = st.session_state.get("edit_country")
    c_data      = q1("SELECT * FROM countries WHERE use_case_id=? AND country=?", ucid, edit_c) if edit_c else {}
    is_new      = not bool(c_data)
    used        = [r["country"] for r in q("SELECT country FROM countries WHERE use_case_id=?", ucid)]
    available   = [c for c in COUNTRIES if c not in used or c == edit_c]

    if edit_c and st.session_state.username != "admin":
        if c_data and not is_country_authorized(st.session_state.username, c_data["id"]):
            st.error("Accès non autorisé.")
            st.stop()

    st.title("Ajouter un pays" if is_new else f"Modifier : {edit_c}")

    cols = get_table_columns("countries")
    extra_cols = [c for c in cols if c not in ("id", "use_case_id", "country")]

    form_values = {}
    with st.form("c_form"):
        st.write("**Identification**")
        country = st.selectbox("Pays *", available, index=available.index(edit_c) if edit_c in available else 0, disabled=not is_new)

        if extra_cols:
            col1, col2 = st.columns(2)
            for i, col_name in enumerate(extra_cols):
                val = c_data.get(col_name, "")
                with col1 if i % 2 == 0 else col2:
                    form_values[col_name] = render_field_input("countries", col_name, val, "co")

        if st.form_submit_button("💾 Sauvegarder", type="primary"):
            data = {"use_case_id": ucid, "country": country}
            for col_name in extra_cols:
                data[col_name] = form_values[col_name]

            if is_new:
                keys = list(data.keys())
                placeholders = ", ".join(["?"] * len(keys))
                sql = f"INSERT INTO countries ({', '.join(keys)}) VALUES ({placeholders})"
                run(sql, *list(data.values()))
                log_change(st.session_state.username, "INSERT", "countries", f"{ucid}/{country}", f"Création du déploiement pays: {country}")
            else:
                keys = [k for k in data.keys() if k not in ("use_case_id", "country")]
                assignments = ", ".join([f"{k}=?" for k in keys])
                values = [data[k] for k in keys] + [ucid, edit_c]
                sql = f"UPDATE countries SET {assignments} WHERE use_case_id=? AND country=?"
                run(sql, *values)
                log_change(st.session_state.username, "UPDATE", "countries", f"{ucid}/{edit_c}", f"Modification du déploiement pays: {edit_c}")

            st.session_state.page = "usecase_detail"
            st.rerun()

    # Column configuration (admin only)
    if st.session_state.username == "admin":
        st.write("---")
        st.subheader("⚙️ Configurer les colonnes de la table Countries")
        st.warning("⚠️ ATTENTION : Supprimer une colonne supprimera définitivement les données correspondantes pour tous les déploiements pays.")

        col_add1, col_add2 = st.columns([3, 1])
        with col_add1:
            new_field = st.text_input("Nom du nouveau champ", placeholder="ex: priorite, contact_local...", key="new_field_countries")
        with col_add2:
            st.write(" ")
            st.write(" ")
            if st.button("➕ Ajouter", key="btn_add_countries"):
                if new_field:
                    sanitized = sanitize_col_name(new_field)
                    existing = get_table_columns("countries")
                    if not sanitized:
                        st.error("Nom de champ invalide.")
                    elif sanitized in existing:
                        st.error("Ce champ existe déjà.")
                    else:
                        run(f"ALTER TABLE countries ADD COLUMN {sanitized} TEXT")
                        log_change(st.session_state.username, "SCHEMA_ADD", "countries", sanitized, f"Ajout de la colonne: {sanitized}")
                        st.success(f"Champ '{sanitized}' ajouté !")
                        st.rerun()
                else:
                    st.error("Veuillez saisir un nom.")

        st.write("**Champs personnalisables existants :**")
        deletable_cols = [c for c in cols if c not in PROTECTED_COLS["countries"]]

        if deletable_cols:
            for c in deletable_cols:
                col_list1, col_list2 = st.columns([3, 1])
                with col_list1:
                    st.code(c)
                with col_list2:
                    if st.button("🗑️ Supprimer", key=f"del_countries_{c}"):
                        run(f"ALTER TABLE countries DROP COLUMN {c}")
                        log_change(st.session_state.username, "SCHEMA_DROP", "countries", c, f"Suppression de la colonne: {c}")
                        st.success(f"Champ '{c}' supprimé !")
                        st.rerun()
        else:
            st.info("Aucun champ personnalisable.")

def render_users_admin_page():
    if st.session_state.username != "admin":
        st.error("Accès réservé à l'administrateur.")
        st.stop()

    st.title("⚙️ Paramètres Habilitations")
    st.subheader("👥 Gestion des utilisateurs et des droits")

    # Fetch all existing country deployments for permission assignments
    deployments = q("""
        SELECT c.id as country_id, f.name as flagship_name, uc.name as usecase_name, c.country as country_name
        FROM countries c
        JOIN use_cases uc ON c.use_case_id = uc.id
        JOIN flagships f ON uc.flagship_id = f.id
        ORDER BY f.rank, uc.id, c.country
    """)

    dep_options = {d["country_id"]: f"{d['flagship_name']} ➔ {d['usecase_name']} ➔ {d['country_name']}" for d in deployments}

    # --- Creation Form ---
    st.write("### ➕ Créer un nouvel utilisateur")
    with st.form("create_user_form"):
        new_u = st.text_input("Nom d'utilisateur", key="adm_new_u")
        new_p = st.text_input("Mot de passe", type="password", key="adm_new_p")

        selected_deps = st.multiselect(
            "Assigner des pays de Use Case de Flagship *",
            list(dep_options.keys()),
            format_func=lambda x: dep_options[x]
        )

        if st.form_submit_button("Créer l'utilisateur", type="primary"):
            if not new_u or not new_p:
                st.error("Veuillez remplir le nom d'utilisateur et le mot de passe.")
            elif new_u == "admin":
                st.error("Le nom d'utilisateur 'admin' est réservé.")
            elif create_user(new_u, new_p):
                # Save assignments
                for cid in selected_deps:
                    run("INSERT INTO user_countries (username, country_id) VALUES (?,?)", new_u, cid)
                st.success(f"Utilisateur '{new_u}' créé avec succès !")
                st.rerun()
            else:
                st.error("Ce nom d'utilisateur existe déjà.")

    st.divider()

    # --- Users List & Modification ---
    st.write("### 👥 Utilisateurs existants")
    users = q("SELECT username FROM users WHERE username != 'admin' ORDER BY username")

    if not users:
        st.info("Aucun autre utilisateur enregistré.")
    else:
        for u in users:
            uname = u["username"]
            with st.expander(f"👤 {uname}"):
                # Get current assigned country IDs
                assigned = [r["country_id"] for r in q("SELECT country_id FROM user_countries WHERE username=?", uname)]

                # Render multiselect with pre-selected values
                new_selected_deps = st.multiselect(
                    f"Modifier les habilitations pour {uname}",
                    list(dep_options.keys()),
                    default=assigned,
                    format_func=lambda x: dep_options[x],
                    key=f"sel_deps_{uname}"
                )

                c_save, c_del = st.columns(2)
                with c_save:
                    if st.button("💾 Enregistrer les modifications", key=f"save_user_{uname}"):
                        # Clear old and write new
                        run("DELETE FROM user_countries WHERE username=?", uname)
                        for cid in new_selected_deps:
                            run("INSERT INTO user_countries (username, country_id) VALUES (?,?)", uname, cid)
                        st.success("Modifications enregistrées !")
                        st.rerun()
                with c_del:
                    if st.button("🗑️ Supprimer cet utilisateur", key=f"del_user_{uname}", type="primary"):
                        run("DELETE FROM users WHERE username=?", uname)
                        run("DELETE FROM user_countries WHERE username=?", uname)
                        st.success("Utilisateur supprimé !")
                        st.rerun()

def render_audit_history_page():
    if st.session_state.username != "admin":
        st.error("Accès réservé à l'administrateur.")
        st.stop()

    st.title("📜 Historique des Actions")
    st.subheader("Journal d'audit des modifications de la plateforme")

    # Clear history button (admin only)
    col_clear, col_spacing = st.columns([1, 3])
    with col_clear:
        if st.button("🗑️ Vider l'historique", type="primary"):
            st.session_state.confirm_clear_history = True

        if st.session_state.get("confirm_clear_history"):
            st.warning("Confirmer le vidage complet ?")
            if st.button("Confirmer", key="confirm_clear_history_btn"):
                run("DELETE FROM audit_logs")
                st.session_state.pop("confirm_clear_history", None)
                st.success("Historique vidé !")
                st.rerun()

    st.write("### 🔍 Filtrer le journal d'audit")
    c_lf1, c_lf2, c_lf3 = st.columns(3)
    with c_lf1:
        all_authors = [r["username"] for r in q("SELECT DISTINCT username FROM audit_logs ORDER BY username")]
        filter_authors = st.multiselect("Filtrer par Auteur", all_authors)
    with c_lf2:
        all_actions = [r["action"] for r in q("SELECT DISTINCT action FROM audit_logs ORDER BY action")]
        filter_actions = st.multiselect("Filtrer par Action", all_actions)
    with c_lf3:
        all_tables = [r["table_name"] for r in q("SELECT DISTINCT table_name FROM audit_logs ORDER BY table_name")]
        filter_tables = st.multiselect("Filtrer par Table", all_tables)

    # Build dynamic query
    sql = "SELECT * FROM audit_logs"
    params = []
    where_clauses = []

    if filter_authors:
        where_clauses.append("username IN ({})".format(", ".join(["?"] * len(filter_authors))))
        params.extend(filter_authors)
    if filter_actions:
        where_clauses.append("action IN ({})".format(", ".join(["?"] * len(filter_actions))))
        params.extend(filter_actions)
    if filter_tables:
        where_clauses.append("table_name IN ({})".format(", ".join(["?"] * len(filter_tables))))
        params.extend(filter_tables)

    if where_clauses:
        sql += " WHERE " + " AND ".join(where_clauses)

    sql += " ORDER BY timestamp DESC"
    logs = q(sql, *params)

    st.write("---")

    if not logs:
        st.info("Aucun historique de modifications correspondant aux critères de filtrage.")
    else:
        for log in logs:
            action_emojis = {
                "INSERT": "➕",
                "UPDATE": "📝",
                "DELETE": "🗑️",
                "SCHEMA_ADD": "⚙️➕",
                "SCHEMA_DROP": "⚙️🗑️"
            }
            emoji = action_emojis.get(log["action"], "🔔")

            # Format nicely
            time_val = log["timestamp"]

            st.markdown(
                f"### {emoji} {log['action']} — Table `{log['table_name']}`\n"
                f"*   **Auteur** : `👤 {log['username']}`\n"
                f"*   **Référence** : `{log['record_id']}`\n"
                f"*   **Détails** : {log['details']}\n"
                f"*   **Date** : 📅 *Le {time_val}*"
            )
            st.write("---")
